In [31]:
import json
import nltk
from rapidfuzz import process, fuzz
from nltk.corpus import stopwords

# Paths
MITRE_DICT_PATH = '/Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/data/processed/MitreDictionary.json'
BING_DATA_PATH = '/Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/data/processed/BingSearchData.json'

# Load MITRE ATT&CK techniques
with open(MITRE_DICT_PATH, 'r') as file:
    mitre_data = json.load(file)

# Load Bing Search Data
with open(BING_DATA_PATH, 'r') as file:
    bing_data = json.load(file)

# Precompiled stop words
STOP_WORDS = set(stopwords.words('english')) - {"ssh", "ftp", "cmd", "rdp", "ssl", "tls", "vpn"}  # Keep security words

# Build searchable text corpus
technique_id_to_text = {
    tech_id: (
        (details.get('name', '') + ' ') +
        (details.get('description', '') + ' ') +
        (' '.join(details.get('keywords', [])) + ' ') * 5  # 🔥 Stronger keyword weight
    ).lower()
    for tech_id, details in mitre_data.items()
}

def preprocess_text(text):
    """
    Preprocess input text: lowercase, selectively remove stopwords, normalize spacing.
    """
    words = text.lower().split()
    filtered_words = [word.strip('.,!?') for word in words if word not in STOP_WORDS and len(word) > 2]
    return ' '.join(filtered_words)

def find_matching_techniques(input_text, threshold=40, top_n=10):
    """
    Find matching MITRE ATT&CK techniques for the given input text using rapid fuzzy matching.
    """
    processed_input = preprocess_text(input_text)
    search_space = list(technique_id_to_text.values())

    matches = process.extract(processed_input, search_space, limit=top_n, scorer=fuzz.token_set_ratio)  # ✅ token_set_ratio

    # Reverse lookup
    text_to_id = {v: k for k, v in technique_id_to_text.items()}

    return [
        (text_to_id[full_text], score)
        for full_text, score, _ in matches
        if score >= threshold
    ]

def extract_unique_tactics(matches):
    """
    Extract tactics with the corresponding matched technique IDs and names.
    """
    tactics_info = []

    for match in matches:
        match_id = match[0]
        technique_info = mitre_data.get(match_id, {})
        technique_name = technique_info.get('name', 'Unknown')
        tactic_list = technique_info.get('tactics', [])

        for tactic in tactic_list:
            tactics_info.append((tactic, match_id, technique_name))
    
    return sorted(tactics_info)

if __name__ == "__main__":
    if not bing_data:
        print("No articles found in Bing search data.")
    else:
        first_article = bing_data[5]  # or 0, or any
        input_text = first_article.get('content', '')
        keywords = first_article.get('keywords', [])

        if not input_text:
            print("No content found in the first article.")
        else:
            combined_input = input_text + ' ' + ' '.join(keywords) * 2  # ✅ Boost keywords

            matches = find_matching_techniques(combined_input)
            tactics_info = extract_unique_tactics(matches)

            print(f"\nCommon MITRE ATT&CK Tactics for Article: {first_article.get('title', 'Unknown Title')}")
            for tactic, match_id, technique_name in tactics_info:
                print(f"- Tactic: {tactic} | Technique: {technique_name} (ID: {match_id})")



Common MITRE ATT&CK Tactics for Article: CrowdStrike Unveils Falcon Privileged Access, Delivering the Only Platform that Unifies End-to-End Hybrid Identity Security
- Tactic: collection | Technique: Data from Information Repositories (ID: T1213)
- Tactic: credential-access | Technique: Steal Application Access Token (ID: T1528)
- Tactic: credential-access | Technique: Hybrid Identity (ID: T1556.007)
- Tactic: defense-evasion | Technique: Cloud Accounts (ID: T1078.004)
- Tactic: defense-evasion | Technique: Trust Modification (ID: T1484.002)
- Tactic: defense-evasion | Technique: Application Access Token (ID: T1527)
- Tactic: defense-evasion | Technique: Application Access Token (ID: T1550.001)
- Tactic: defense-evasion | Technique: Hybrid Identity (ID: T1556.007)
- Tactic: impact | Technique: Endpoint Denial of Service (ID: T1499)
- Tactic: initial-access | Technique: Cloud Accounts (ID: T1078.004)
- Tactic: initial-access | Technique: Spearphishing Link (ID: T1566.002)
- Tactic: late